# Walkthrough: Carbon Capture in Synthetic Biology

This notebook is an end-to-end demonstration of the full analysis pipeline,
focused on the **carbon capture case study**.

It is designed to be readable on the project website as a reproducibility
companion. Each step shows what is happening and why.

**Steps covered:**
1. Load and inspect the processed datasets
2. Filter to the carbon-capture subset
3. Explore the semantic space (UMAP projection)
4. Examine cluster composition
5. Look at city-level patterns
6. Timeline of carbon-capture activity

**Before running:** complete notebooks 01–06 to generate all data files.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
from pathlib import Path

PROCESSED = Path('../data/processed')
EMBEDDINGS = Path('../data/embeddings')

In [ ]:
# --- 1. Load all processed datasets ---

dfs = {}
for name in ['papers', 'patents', 'projects', 'parts']:
    path = PROCESSED / f'{name}.csv'
    if path.exists():
        dfs[name] = pd.read_csv(path)
        print(f'{name}: {len(dfs[name])} records')

all_artifacts = pd.concat(list(dfs.values()), ignore_index=True)
print(f'\nTotal: {len(all_artifacts)} artifacts')

In [ ]:
# --- 2. Filter to carbon-capture subset ---

cc = all_artifacts[all_artifacts['case_study_flag'] == True].copy()

print(f'Carbon capture artifacts: {len(cc)}')
print(cc['type'].value_counts())

In [ ]:
# --- 3. Load UMAP projections ---

import json

proj_path = EMBEDDINGS / 'projections.json'
if proj_path.exists():
    with open(proj_path) as f:
        projections = pd.DataFrame(json.load(f))
    
    # Merge with metadata
    cc_proj = cc.merge(projections, on='id', how='inner')
    all_proj = all_artifacts.merge(projections, on='id', how='inner')
    print(f'Projections loaded: {len(projections)} entries')
else:
    print('projections.json not found. Run notebook 05 first.')

In [ ]:
# --- 4. Semantic space: all artifacts, carbon capture highlighted ---

if 'all_proj' in dir():
    all_proj['highlight'] = all_proj['id'].isin(cc['id'])
    all_proj['point_label'] = all_proj['highlight'].map(
        {True: 'Carbon capture', False: 'Other'}
    )

    fig = px.scatter(
        all_proj,
        x='x', y='y',
        color='point_label',
        color_discrete_map={'Carbon capture': '#e15759', 'Other': '#aaa'},
        opacity=0.6,
        hover_data=['title', 'type', 'year'],
        title='Semantic Space — Carbon Capture Highlighted',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    )
    fig.update_traces(marker_size=4)
    fig.show()

In [ ]:
# --- 5. Timeline: carbon capture activity by year and artifact type ---

cc_by_year = (
    cc.groupby(['year', 'type'])
    .size()
    .reset_index(name='count')
)

fig = px.bar(
    cc_by_year,
    x='year', y='count', color='type',
    barmode='stack',
    title='Carbon Capture Activity by Year and Artifact Type',
    labels={'year': 'Year', 'count': 'Number of Artifacts'},
)
fig.show()

In [ ]:
# --- 6. City-level summary ---

cc_cities = (
    cc.groupby(['city', 'country', 'type'])
    .size()
    .reset_index(name='count')
    .pivot_table(index=['city', 'country'], columns='type', values='count', fill_value=0)
    .reset_index()
)

print('Top cities for carbon capture:')
cc_cities['total'] = cc_cities.drop(columns=['city', 'country'], errors='ignore').sum(axis=1)
print(cc_cities.sort_values('total', ascending=False).head(15))